In [0]:
%sql

SELECT * FROM 4_prod.pacs.imaging_metadata
LIMIT 100

In [0]:
%sql
SELECT ExaminationModality, YEAR(MillEventDate) AS year, COUNT(DISTINCT clinical_event_id) FROM 4_prod.pacs.imaging_metadata
WHERE YEAR(MillEventDate) IS NOT NULL
GROUP BY ExaminationModality, YEAR(MillEventDate)
HAVING COUNT(DISTINCT clinical_event_id) > 50
ORDER BY ExaminationModality, Year(MillEventDate) ASC


In [0]:
%sql
WITH modality_year_counts AS (
  SELECT 
    ExaminationModality, 
    YEAR(MillEventDate) AS year, 
    COUNT(DISTINCT AccessionNbr) AS accession_count
  FROM 4_prod.pacs.imaging_metadata
  WHERE YEAR(MillEventDate) IS NOT NULL
  GROUP BY ExaminationModality, YEAR(MillEventDate)
  HAVING COUNT(DISTINCT AccessionNbr) > 50
),
filtered_data AS (
  SELECT 
    m.ExaminationModality,
    YEAR(m.MillEventDate) AS year,
    m.AccessionNbr
  FROM 4_prod.pacs.imaging_metadata m
  INNER JOIN modality_year_counts c
    ON m.ExaminationModality = c.ExaminationModality
    AND YEAR(m.MillEventDate) = c.year
  WHERE m.AccessionNbr IS NOT NULL
)
SELECT 
  ExaminationModality,
  year,
  AccessionNbr
FROM (
  SELECT 
    ExaminationModality,
    year,
    AccessionNbr,
    ROW_NUMBER() OVER (PARTITION BY ExaminationModality, year ORDER BY rand()) AS rn
  FROM filtered_data
) t
WHERE rn <= 10
ORDER BY ExaminationModality, year, rn